In [4]:
import numpy as np
import parselmouth
from parselmouth.praat import call
from typing import Dict, List

In [5]:
def compute_spectral_proximity(file_path: str) -> Dict[str, float]:
    """
    Hypothesis 1: Tracks F0 and F1 to calculate the average distance.

    Args:
        file_path (str): -> Absolute or relative path to a .wav audio file.

    Returns:
        Dict[str, float]: -> A dictionary containing absolute and normalized metrics.
                          Values will be np.nan if tracking fails.
    """
    # Initialize your default dictionary layout
    results = {
        "spectral_proximity_abs_hz": np.nan,
        "spectral_proximity_norm_ratio": np.nan
    }
    try:
        sound = parselmouth.Sound(file_path)
        pitch = call(sound, "To Pitch", 0.0, 75.0, 600.0)
        formant_object = call(sound, "To Formant (burg)", 0.0, 5, 5000.0, 0.025, 50.0)
        num_formant_frames = call(formant_object, "Get number of frames")

        abs_distances = []
        norm_distances = []
        
        for frame in range(1, num_formant_frames + 1):
            current_time_sec = call(formant_object, "Get time from frame number", frame)
            f1 = call(formant_object, "Get value at time", 1, current_time_sec, "Hertz", "Linear")
            f0 = call(pitch, "Get value at time", current_time_sec, "Hertz", "Linear")
            
            if not np.isnan(f0) and not np.isnan(f1) and f0 > 0:
                abs_distances.append(f1 - f0)
                norm_distances.append((f1 - f0) / f0)
            
            # update values if valid frames were successfully calculated
            if len(abs_distances) > 0:
                results["spectral_proximity_abs_hz"] = float(np.mean(abs_distances))
                results["spectral_proximity_norm_ratio"] = float(np.mean(norm_distances))
                
            
    except Exception as e:
        print(f"⚠️ Tracking exception inside compute_spectral_proximity: {e}")
    
    return results
    
    

In [21]:
from typing import Dict
import numpy as np
import parselmouth
from parselmouth.praat import call

def compute_formant_clustering(file_path: str) -> Dict[str, float]:
    """Hypothesis 2: Quantifies formant clustering in the 2000-4000 Hz region
    using LTAS energy balance and formant dispersion tracking.

    Args:
        file_path (str): path to a .wav file.

    Returns:
        Dict[str, float]: Dictionary containing energy ratio and dispersion features.
    """
    results = {
        "clustering_ltas_ratio_db": np.nan,
        "clustering_ltas_max_peak_db": np.nan,
        "clustering_f3_f4_dispersion_hz": np.nan
    }
    
    try:
        sound = parselmouth.Sound(file_path)
        # --- Feature 1: LTAS Energy Balance ---
        # Convert to Long-Term Average Spectrum (bandwidth resolution 100Hz)
        ltas = call(sound, "To Ltas", 100.0)
        energy_low = call(ltas, "Get mean", 0.0, 2000.0, "dB")
        energy_high = call(ltas, "Get mean", 2000.0, 4000.0, "dB")
        max_peak_high = call(ltas, "Get maximum", 2000.0, 4000.0, "Parabolic")
        # Check the raw values driving the subtraction
        print("--- 🔍 Raw Band Energy Audit ---")
        print(f"Low (0-2kHz): {energy_low:.2f} dB | High (2k-4kHz): {energy_high:.2f} dB")
     
        # In dB, subtraction equals the true division ratio
        results["clustering_ltas_ratio_db"] = float(energy_high-energy_low)
        results["clustering_ltas_max_peak_db"] = float(max_peak_high)
        
        # --- Feature 2: Formant Line Dispersion ---
        # Track formants frame-by-frame
        formant_object = call(sound, "To Formant (burg)", 0.0, 5, 5000.0, 0.025, 50.0)
        num_frames = call(formant_object, "Get number of frames")
        
        frame_distances = []
        for frame in range(1, num_frames + 1):
            time = call(formant_object, "Get time from frame number", frame)
            f3 = call(formant_object, "Get value at time", 3, time, "Hertz", "Linear")
            f4 = call(formant_object, "Get value at time", 4, time, "Hertz", "Linear")
            
            if not np.isnan(f3) and not np.isnan(f4):
                # Calculate the localized gap between F3 and F4
                frame_distances.append(abs(f4 - f3))
            if len(frame_distances) > 0:
                results["clustering_f3_f4_dispersion_hz"] = float(np.mean(frame_distances))
                
    except Exception as e:
        print(f"⚠️ Exception inside compute_formant_clustering: {e}")
    
    return results

In [22]:
# Notebook diagnostic cell
resonant_sample = "../data/processed/M062_Mic_a_resonant_1st.wav"
non_resonant_sample = "../data/processed/M062_Mic_a_non-resonant_1st.wav"

print("Resonant")
res_metrics = compute_formant_clustering(resonant_sample)
print("Non-Resonant")
non_res_metrics = compute_formant_clustering(non_resonant_sample)

print("--- 📊 Hypothesis 2 Diagnostic Verification ---")
print(f"Resonant     -> LTAS Ratio: {res_metrics['clustering_ltas_ratio_db']:.2f} dB | Max Peak: {res_metrics['clustering_ltas_max_peak_db']: .1f}| F3-F4 Dispersion: {res_metrics['clustering_f3_f4_dispersion_hz']:.1f} Hz")
print(f"Non-Resonant -> LTAS Ratio: {non_res_metrics['clustering_ltas_ratio_db']:.2f} dB | Max Peak: {non_res_metrics['clustering_ltas_max_peak_db']: .1f} | F3-F4 Dispersion: {non_res_metrics['clustering_f3_f4_dispersion_hz']:.1f} Hz")

Resonant
--- 🔍 Raw Band Energy Audit ---
Low (0-2kHz): 40.02 dB | High (2k-4kHz): 20.41 dB
Non-Resonant
--- 🔍 Raw Band Energy Audit ---
Low (0-2kHz): 40.03 dB | High (2k-4kHz): 21.71 dB
--- 📊 Hypothesis 2 Diagnostic Verification ---
Resonant     -> LTAS Ratio: -19.61 dB | Max Peak:  33.2| F3-F4 Dispersion: 509.1 Hz
Non-Resonant -> LTAS Ratio: -18.32 dB | Max Peak:  34.1 | F3-F4 Dispersion: 776.9 Hz


In [ ]:
def compute_micro_modulations(file_path: str) -> Dict[str, float]:
    """Hypothesis 3: Evaluates frame-to-frame local differences in both F1 amplitude 
    (shimmer-like) and F1 frequency (jitter-like), gated by pitch tracking.

    Args:
        file_path (str): Path to a .wav audio file.

    Returns:
        Dict[str, float]: Dictionary containing amplitude and frequency perturbation standard deviations.
    """